# Week 08A — Object Detection I: Fundamentals and Two-Stage Detectors

**MCTA 4364 Machine Vision** · Session 1 of Week 8 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W08_detection/W08A_detection_fundamentals.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W08_detection/W08A_detection_fundamentals.ipynb)

Classification answers *"what is in the image?"*. **Detection** answers *"what is where?"*: a list of **boxes**, each with a **class** and a
**confidence**. It is the workhorse of machine vision: counting parts, picking objects with a robot, checking that all components are present,
monitoring people and vehicles. This session builds detection from first principles — boxes, IoU, sliding windows, non-maximum suppression and
average precision — then opens up a modern **two-stage** detector (Faster R-CNN) to see the same ideas implemented with a CNN.

### Learning outcomes
By the end of this session you will be able to:
1. Represent boxes in the VOC, COCO and YOLO formats and convert between them.
2. Compute **IoU** and explain why small objects are hard to localise.
3. Build a classical **sliding-window detector** (HOG + linear SVM) and explain its cost.
4. Implement **non-maximum suppression** and compute **precision–recall curves** and **average precision (AP / mAP)**.
5. Explain anchors, the region proposal network, RoIAlign and FPN in **Faster R-CNN**, and inspect its two stages.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1–2 | The detection task; box and annotation formats |
| 8–18 | 3 | IoU (interactive) |
| 18–35 | 4–5 | Sliding-window HOG detector; NMS |
| 35–50 | 6 | Precision, recall, AP and mAP (interactive) |
| 50–72 | 7 | Two-stage detectors: anchors, RPN, RoIAlign, FPN; Faster R-CNN inside out |
| 72–80 | 10 | Quiz and exit ticket |
| Home | 8–9 | Exercises with self-checks, challenge |

**Hardware:** runs on a laptop CPU (about 2 minutes); the Faster R-CNN section downloads ~75 MB of weights once.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from skimage.feature import hog

import partsim
from cvhelpers import SMOKE, check, interact, load_sample, opencv_sample, pipeline_diagram, quiz, show
from dlhelpers import hw_profile, load_pretrained, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
set_seed(0)
NAMES = partsim.SCENE_CLASSES
N_TRAIN_SCENES, N_TEST_SCENES = (6, 4) if SMOKE else (40, 20)
scene, gt_boxes, gt_labels, gt_masks = partsim.make_scene(4, return_masks=True)
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi")))
_, street = cap.read(); cap.release()
messi = load_sample("messi5.jpg")
show(partsim.draw_boxes(scene, gt_boxes, gt_labels), street, messi,
     titles=["synthetic conveyor scene with ground-truth boxes (partsim)", "vtest.avi (OpenCV sample)", "messi5.jpg (OpenCV sample)"], figsize=(22, 5.5))

## 1. The detection task

| task | output | example question |
|---|---|---|
| classification | one label per image | *Is there a bolt in the tray?* |
| localisation | one label + one box | *Where is the bolt?* |
| **detection** | a **variable-length list** of (box, class, score) | *Where are all the nuts, washers and bolts?* |
| instance segmentation | a list of (mask, class, score) | *Which pixels belong to each part?* (Week 11) |

The variable-length output is what makes detection hard: the network does not know in advance how many objects there are.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
cls_counts = {NAMES[i]: int((gt_labels == i).sum()) for i in range(len(NAMES))}
axes[0].imshow(scene[..., ::-1]); axes[0].set_title("classification: " + ", ".join(k for k, v in cls_counts.items() if v))
big = int(np.argmax([(b[2] - b[0]) * (b[3] - b[1]) for b in gt_boxes]))
axes[1].imshow(partsim.draw_boxes(scene, gt_boxes[big:big + 1], gt_labels[big:big + 1])[..., ::-1]); axes[1].set_title("localisation: one object")
axes[2].imshow(partsim.draw_boxes(scene, gt_boxes, gt_labels)[..., ::-1]); axes[2].set_title(f"detection: {len(gt_boxes)} objects")
overlay = scene.copy()
rng = np.random.default_rng(0)
for m_ in gt_masks:
    overlay[m_] = (0.4 * overlay[m_] + 0.6 * rng.integers(60, 255, 3)).astype(np.uint8)
axes[3].imshow(overlay[..., ::-1]); axes[3].set_title("instance segmentation (Week 11)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.suptitle("Figure 1.1 - From classification to instance segmentation on the same image", fontweight="bold", y=1.03)
plt.show()

## 2. Boxes and annotation formats

The same box can be written in three common ways. Mixing them up is the most frequent bug in detection projects.

| format | used by | box | units |
|---|---|---|---|
| **xyxy** | Pascal VOC, torchvision, most maths | $(x_{min}, y_{min}, x_{max}, y_{max})$ | pixels |
| **xywh** | **COCO** JSON | $(x_{min}, y_{min}, w, h)$ | pixels |
| **cxcywh normalised** | **YOLO** `.txt` (one line per object: `class cx cy w h`) | centre and size divided by image width/height | 0–1 |

In [ ]:
H_, W_ = scene.shape[:2]
b = gt_boxes[0]
xywh = [b[0], b[1], b[2] - b[0], b[3] - b[1]]
yolo = [(b[0] + b[2]) / 2 / W_, (b[1] + b[3]) / 2 / H_, (b[2] - b[0]) / W_, (b[3] - b[1]) / H_]
print(f"class {gt_labels[0]} ({NAMES[gt_labels[0]]})")
print("xyxy (VOC / torchvision):", np.round(b, 1))
print("xywh (COCO):             ", np.round(xywh, 1))
print("YOLO line:               ", gt_labels[0], " ".join(f"{v:.4f}" for v in yolo))
print("torchvision.ops.box_convert agrees:", torchvision.ops.box_convert(torch.tensor(b[None]), "xyxy", "cxcywh").numpy().round(1))

## 3. Intersection over Union (IoU)

$$\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} \in [0, 1]$$

IoU decides whether a prediction **matches** a ground-truth object (usually IoU ≥ 0.5), and whether two predictions are **duplicates** (in NMS).
Because it is a *ratio*, the same pixel error hurts a small object far more than a large one.

In [ ]:
def box_iou(a, b):
    """IoU between every box in a [N, 4] and every box in b [M, 4] (xyxy). Returns [N, M]."""
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(rb - lt, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-12)


fig, axes = plt.subplots(1, 2, figsize=(16, 4.3))
shifts = np.arange(0, 41)
for size, col in [(20, "#d93025"), (50, "#f29900"), (100, "#188038"), (300, "#1a73e8")]:
    g = np.array([0, 0, size, size])
    axes[0].plot(shifts, [box_iou(g, g + [s, 0, s, 0])[0, 0] for s in shifts], lw=2.5, color=col, label=f"{size} x {size} px object")
axes[0].axhline(0.5, color="k", ls="--"); axes[0].set_xlabel("horizontal localisation error (px)"); axes[0].set_ylabel("IoU")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("a 10 px error is fatal for a 20 px object, harmless for 300 px")
examples = [(0, "IoU 1.0"), (8, ""), (20, ""), (35, "")]
g = np.array([30, 30, 130, 130])
axes[1].set_xlim(0, 480); axes[1].set_ylim(170, 0); axes[1].set_aspect("equal"); axes[1].axis("off")
for k, (s, _) in enumerate(examples):
    off = k * 115
    axes[1].add_patch(plt.Rectangle((g[0] + off - 20, g[1]), 100, 100, fill=False, ec="#188038", lw=2.5))
    p = g + [s + off - 20, s // 2, s + off - 20, s // 2]
    axes[1].add_patch(plt.Rectangle((p[0], p[1]), 100, 100, fill=False, ec="#d93025", lw=2, ls="--"))
    axes[1].text(g[0] + off + 30, 160, f"IoU = {box_iou(g, g + [s, s // 2, s, s // 2])[0, 0]:.2f}", ha="center")
axes[1].set_title("green = ground truth, red = prediction")
plt.suptitle("Figure 3.1 - IoU and object size", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: move and resize a prediction

In [ ]:
def iou_demo(dx=10, dy=5, scale=1.0):
    g = np.array([150, 100, 330, 260], float)
    c = np.array([(g[0] + g[2]) / 2 + dx, (g[1] + g[3]) / 2 + dy]); wh = np.array([g[2] - g[0], g[3] - g[1]]) * scale
    p = np.r_[c - wh / 2, c + wh / 2]
    iou = box_iou(g, p)[0, 0]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.add_patch(plt.Rectangle(g[:2], *(g[2:] - g[:2]), fc="#188038", alpha=0.25, ec="#188038", lw=2, label="ground truth"))
    ax.add_patch(plt.Rectangle(p[:2], *(p[2:] - p[:2]), fc="#d93025", alpha=0.25, ec="#d93025", lw=2, label="prediction"))
    ax.set_xlim(0, 480); ax.set_ylim(360, 0); ax.set_aspect("equal"); ax.legend(loc="lower right")
    ax.set_title(f"IoU = {iou:.3f} -> {'TRUE positive' if iou >= 0.5 else 'FALSE positive'} at threshold 0.5")
    plt.show()


interact(iou_demo, dx=widgets.IntSlider(value=10, min=-150, max=150), dy=widgets.IntSlider(value=5, min=-150, max=150),
         scale=widgets.FloatSlider(value=1.0, min=0.3, max=2.0, step=0.05))

## 4. A classical detector: sliding window + HOG + linear SVM

Before deep learning, detection was **classification applied everywhere** (Dalal & Triggs, 2005, for pedestrians; Viola & Jones, 2001, for faces):

1. slide a fixed-size **window** over the image (and over an **image pyramid** for other scales, Week 5A);
2. describe each window with a **Histogram of Oriented Gradients** (HOG): gradient-orientation histograms in small cells, normalised over blocks;
3. score it with a **linear SVM** $s = \mathbf{w}^\top \mathbf{x} + b$;
4. keep windows with $s > \tau$ and remove duplicates with **NMS** (Section 5).

We build one for the `hex_nut` class. The linear SVM is trained with PyTorch: a linear layer with the **hinge loss** $\max(0, 1 - y s)$.

In [ ]:
CELL, WIN_CELLS = 6, 7                       # 6 px cells, 7 x 7-cell windows = 42 x 42 px (a nut is ~34-38 px)
WIN = CELL * WIN_CELLS
TARGET = NAMES.index("hex_nut")


def gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255


def hog_windows(img):
    """HOG of the whole image once, then every window position (stride = 1 cell). Returns (features [N, D], boxes [N, 4])."""
    H = hog(gray(img), orientations=9, pixels_per_cell=(CELL, CELL), cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=False)
    nb = WIN_CELLS - 1                        # blocks per window side
    ny, nx = H.shape[0] - nb + 1, H.shape[1] - nb + 1
    feats = np.lib.stride_tricks.sliding_window_view(H, (nb, nb), axis=(0, 1))       # [ny, nx, 2, 2, 9, nb, nb]
    feats = feats.transpose(0, 1, 5, 6, 2, 3, 4).reshape(ny * nx, -1).astype(np.float32)
    gy, gx = np.mgrid[0:ny, 0:nx]
    boxes = np.stack([gx.ravel() * CELL, gy.ravel() * CELL, gx.ravel() * CELL + WIN, gy.ravel() * CELL + WIN], 1).astype(np.float32)
    return feats, boxes


crop_pos = scene[int(gt_boxes[gt_labels == TARGET][0][1]):, int(gt_boxes[gt_labels == TARGET][0][0]):][:WIN, :WIN]
crop_neg = scene[10:10 + WIN, 10:10 + WIN]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for k, (crop, name) in enumerate([(crop_pos, "nut window"), (crop_neg, "belt window")]):
    _, vis = hog(gray(crop), orientations=9, pixels_per_cell=(CELL, CELL), cells_per_block=(2, 2), visualize=True)
    axes[2 * k].imshow(crop[..., ::-1]); axes[2 * k].set_title(name); axes[2 * k + 1].imshow(vis, cmap="magma"); axes[2 * k + 1].set_title("its HOG")
for ax in axes:
    ax.axis("off")
feats, wboxes = hog_windows(scene)
plt.suptitle(f"Figure 4.1 - HOG describes shape by local gradient orientations ({feats.shape[1]} numbers per window, {len(feats)} windows per image)",
             fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def collect(n_scenes, seed, neg_fraction=0.05):
    """Positives: windows with IoU > 0.6 with a nut. Negatives: a random sample of windows with IoU < 0.3 with every nut."""
    rng = np.random.default_rng(seed)
    P, N = [], []
    for _ in range(n_scenes):
        img, b, l = partsim.make_scene(rng.integers(1 << 31))
        F, B = hog_windows(img)
        best = box_iou(B, b[l == TARGET]).max(1) if (l == TARGET).any() else np.zeros(len(B))
        P.append(F[best > 0.6])
        neg = np.nonzero(best < 0.3)[0]
        N.append(F[rng.choice(neg, int(len(neg) * neg_fraction), replace=False)])
    return np.vstack(P), np.vstack(N)


def train_linear_svm(P, N, epochs=300, lam=1e-3):
    X = torch.tensor(np.vstack([P, N]))
    y = torch.tensor(np.r_[np.ones(len(P)), -np.ones(len(N))], dtype=torch.float32)
    weight = torch.where(y > 0, len(N) / len(P), 1.0)             # balance the rare positives
    lin = torch.nn.Linear(X.shape[1], 1)
    opt = torch.optim.Adam(lin.parameters(), 1e-2)
    for _ in range(epochs):
        opt.zero_grad()
        s = lin(X).squeeze(1)
        loss = (weight * torch.clamp(1 - y * s, min=0)).mean() + lam * (lin.weight ** 2).sum()
        loss.backward(); opt.step()
    return lin.weight.detach().numpy().ravel(), float(lin.bias.detach())


t0 = time.perf_counter()
P, N = collect(N_TRAIN_SCENES, seed=1)
w_svm, b_svm = train_linear_svm(P, N)
print(f"{len(P)} positive and {len(N)} negative windows from {N_TRAIN_SCENES} scenes; trained in {time.perf_counter() - t0:.1f} s")

In [ ]:
test_img, test_b, test_l = partsim.make_scene(1234)
F, B = hog_windows(test_img)
scores = F @ w_svm + b_svm
ny = (test_img.shape[0] // CELL) - WIN_CELLS + 1
score_map = scores.reshape(ny, -1)
fig, axes = plt.subplots(1, 3, figsize=(22, 5.2))
axes[0].imshow(test_img[..., ::-1]); axes[0].set_title("test scene")
im_ = axes[1].imshow(score_map, cmap="RdBu_r", vmin=-3, vmax=3, extent=[WIN / 2, WIN / 2 + score_map.shape[1] * CELL, WIN / 2 + ny * CELL, WIN / 2])
plt.colorbar(im_, ax=axes[1], fraction=0.035); axes[1].set_title("SVM score at every window centre")
raw = B[scores > 0]
axes[2].imshow(partsim.draw_boxes(test_img, raw, np.full(len(raw), TARGET), thickness=1)[..., ::-1])
axes[2].set_title(f"{len(raw)} windows with score > 0: duplicates everywhere")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 4.2 - Sliding-window detection: a classifier evaluated at every position", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Non-maximum suppression (NMS)

Every object fires many overlapping windows. **Greedy NMS** keeps the best and removes its neighbours:

1. sort the boxes by score;
2. take the highest-scoring box, keep it;
3. remove every remaining box with IoU > $t_\text{NMS}$ (e.g. 0.3–0.7) with it;
4. repeat until no boxes remain.

Modern detectors still use NMS (YOLO up to v10 made it optional, DETR removes it — Session B). *Soft-NMS* decays scores instead of deleting boxes,
which helps with crowded, overlapping objects.

In [ ]:
def nms_steps(boxes, scores, thr):
    order = list(np.argsort(-scores)); steps = []
    while order:
        i = order.pop(0)
        ious = box_iou(boxes[i], boxes[order])[0] if order else np.array([])
        removed = [order[k] for k in np.nonzero(ious > thr)[0]]
        order = [order[k] for k in np.nonzero(ious <= thr)[0]]
        steps.append((i, removed))
    return steps


k_ = scores > 0
Bk, Sk = B[k_], scores[k_]
steps = nms_steps(Bk, Sk, 0.3)
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, n_steps in zip(axes, [0, 1, 3, len(steps)]):
    kept = [s[0] for s in steps[:n_steps]]
    gone = set(sum([s[1] for s in steps[:n_steps]], []))
    rest = [i for i in range(len(Bk)) if i not in kept and i not in gone]
    vis = test_img.copy()
    for i in rest:
        cv2.rectangle(vis, tuple(Bk[i, :2].astype(int)), tuple(Bk[i, 2:].astype(int)), (160, 160, 160), 1)
    for i in kept:
        cv2.rectangle(vis, tuple(Bk[i, :2].astype(int)), tuple(Bk[i, 2:].astype(int)), (0, 200, 0), 2)
    ax.imshow(vis[..., ::-1]); ax.axis("off")
    ax.set_title(f"after {n_steps} steps: {len(kept)} kept, {len(gone)} suppressed, {len(rest)} left")
plt.suptitle("Figure 5.1 - Greedy NMS (IoU threshold 0.3): green = kept, grey = still to process", fontweight="bold")
plt.tight_layout(); plt.show()
keep_tv = torchvision.ops.nms(torch.tensor(Bk), torch.tensor(Sk), 0.3).numpy()
print("our NMS keeps", len(steps), "boxes; torchvision.ops.nms keeps", len(keep_tv))

## 6. Evaluating a detector: precision, recall, AP and mAP

Sort all detections (over the whole test set) by score. Walking down the list, each detection is a **true positive** if it matches an unmatched
ground-truth object with IoU ≥ 0.5, otherwise a **false positive** (a second detection of the same object is a false positive too). At each rank:

$$\text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{\#\text{ground-truth objects}}.$$

**Average precision (AP)** is the area under the precision–recall curve, after making precision monotonically decreasing (the "envelope").
**mAP** averages AP over classes; **COCO mAP** (written mAP@[.5:.95] or mAP50-95) also averages over IoU thresholds 0.50, 0.55, …, 0.95 and therefore
rewards accurate localisation. `mAP50` uses IoU 0.5 only.

In [ ]:
def detect_nuts(img, thr=-1.0, nms_thr=0.3):
    F_, B_ = hog_windows(img)
    s_ = F_ @ w_svm + b_svm
    k = s_ > thr
    if not k.any():
        return np.zeros((0, 4), np.float32), np.zeros(0, np.float32)
    keep = torchvision.ops.nms(torch.tensor(B_[k]), torch.tensor(s_[k]), nms_thr).numpy()
    return B_[k][keep], s_[k][keep]


def match_detections(det_boxes, det_scores, gts, iou_thr=0.5):
    """Greedy matching in score order. Returns a boolean array: is each detection a true positive?"""
    tp = np.zeros(len(det_boxes), bool)
    if len(gts) == 0 or len(det_boxes) == 0:
        return tp
    ious = box_iou(det_boxes, gts)
    used = np.zeros(len(gts), bool)
    for i in np.argsort(-det_scores):
        j = int(np.argmax(np.where(used, -1, ious[i])))
        if ious[i, j] >= iou_thr and not used[j]:
            tp[i] = True; used[j] = True
    return tp


rng = np.random.default_rng(99)
test_set = [partsim.make_scene(rng.integers(1 << 31)) for _ in range(N_TEST_SCENES)]
all_dets = [detect_nuts(img) for img, _, _ in test_set]
n_gt = sum(int((l == TARGET).sum()) for _, _, l in test_set)


def pr_curve(iou_thr):
    s_all, tp_all = [], []
    for (img, b, l), (D, S) in zip(test_set, all_dets):
        s_all.append(S); tp_all.append(match_detections(D, S, b[l == TARGET], iou_thr))
    s_all, tp_all = np.concatenate(s_all), np.concatenate(tp_all)
    order = np.argsort(-s_all)
    tp_c, fp_c = np.cumsum(tp_all[order]), np.cumsum(~tp_all[order])
    return tp_c / max(n_gt, 1), tp_c / np.maximum(tp_c + fp_c, 1), s_all[order]


def ap_from_pr(rec, prec):
    """All-point interpolated AP (VOC 2010+): area under the precision envelope."""
    mrec, mpre = np.r_[0, rec, rec[-1] if len(rec) else 0], np.r_[1, prec, 0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    return float(np.sum(np.diff(mrec) * mpre[1:]))


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
aps = {}
for thr_, col in [(0.5, "#1a73e8"), (0.75, "#d93025")]:
    rec, prec, _ = pr_curve(thr_)
    aps[thr_] = ap_from_pr(rec, prec)
    env = np.maximum.accumulate(prec[::-1])[::-1]
    axes[0].plot(rec, prec, color=col, alpha=0.4); axes[0].plot(rec, env, color=col, lw=2.5, label=f"IoU {thr_}: AP = {aps[thr_]:.3f}")
axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision"); axes[0].set_xlim(0, 1.02); axes[0].set_ylim(0, 1.02); axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].set_title(f"hex_nut detector on {N_TEST_SCENES} test scenes ({n_gt} nuts)")
ts = np.arange(0.5, 0.96, 0.05)
ap_t = [ap_from_pr(*pr_curve(t)[:2]) for t in ts]
axes[1].bar([f"{t:.2f}" for t in ts], ap_t, color="#5f6368"); axes[1].set_xlabel("IoU threshold"); axes[1].set_ylabel("AP")
axes[1].set_title(f"COCO-style AP@[.5:.95] = mean of these = {np.mean(ap_t):.3f}")
plt.suptitle("Figure 6.1 - Precision-recall curves and average precision (thin: raw, thick: envelope)", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: the operating point
A deployed detector uses one score threshold. Move it and see the detections and where you sit on the PR curve.

In [ ]:
rec50, prec50, sc50 = pr_curve(0.5)


def threshold_demo(threshold=1.0):
    img, b, l = test_set[0]
    D, S = all_dets[0]
    k = S > threshold
    tp = match_detections(D[k], S[k], b[l == TARGET])
    vis = img.copy()
    for g in b[l == TARGET]:
        cv2.rectangle(vis, tuple(g[:2].astype(int)), tuple(g[2:].astype(int)), (255, 255, 255), 1)
    for d, t in zip(D[k], tp):
        cv2.rectangle(vis, tuple(d[:2].astype(int)), tuple(d[2:].astype(int)), (0, 200, 0) if t else (0, 0, 255), 2)
    idx = np.searchsorted(-sc50, -threshold)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw=dict(width_ratios=[1.4, 1]))
    axes[0].imshow(vis[..., ::-1]); axes[0].axis("off"); axes[0].set_title("white = ground truth, green = TP, red = FP")
    axes[1].plot(rec50, prec50, color="#1a73e8")
    if idx > 0:
        axes[1].plot(rec50[idx - 1], prec50[idx - 1], "o", ms=12, color="#d93025")
        axes[1].set_title(f"threshold {threshold:.2f}: precision {prec50[idx - 1]:.2f}, recall {rec50[idx - 1]:.2f}")
    axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_xlim(0, 1.02); axes[1].set_ylim(0, 1.02); axes[1].grid(alpha=0.3)
    plt.show()


interact(threshold_demo, threshold=widgets.FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1))

## 7. Two-stage deep detectors: R-CNN → Fast R-CNN → Faster R-CNN

The sliding-window detector wastes most of its time on background and uses a hand-made feature. The **R-CNN family** (Girshick et al.) replaced both:

| model | proposals | features | speed |
|---|---|---|---|
| R-CNN (2014) | ~2000 from *selective search* (classical segmentation) | a CNN run **on each proposal** | ~50 s / image |
| Fast R-CNN (2015) | selective search | **one** CNN pass; **RoI pooling** crops each proposal from the feature map | ~2 s |
| **Faster R-CNN** (2015) | a **Region Proposal Network** (RPN) on the same features | shared | ~0.1 s on a GPU |
| + **FPN** (2017) | proposals on a **feature pyramid** (Week 5A pyramids, learned) | multi-scale | small objects much better |

Key ideas:
- **Anchors**: at every feature-map position (stride 16 or 32 px) the RPN considers $k$ reference boxes of several scales and aspect ratios, and predicts
  (i) an *objectness* score and (ii) 4 offsets that refine each anchor. **Stage 1** = keep the top proposals after NMS.
- **RoIAlign** (Mask R-CNN, 2017): sample a fixed-size (e.g. 7×7) feature grid from each proposal with **bilinear interpolation** (Week 3A!) instead of
  rounding, so small objects are localised accurately.
- **Stage 2**: a small head classifies each RoI into $C + 1$ classes (including *background*) and refines its box again, then per-class NMS.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
stride, scales, ratios = 32, [64, 128, 256], [0.5, 1.0, 2.0]
axes[0].imshow(messi[..., ::-1])
for y in range(0, messi.shape[0], stride):
    axes[0].axhline(y, color="white", lw=0.3, alpha=0.5)
for x in range(0, messi.shape[1], stride):
    axes[0].axvline(x, color="white", lw=0.3, alpha=0.5)
cols = {64: "#fbbc05", 128: "#34a853", 256: "#ea4335"}
for (cx, cy) in [(368, 144), (144, 208)]:
    for s in scales:
        for r in ratios:
            w, h = s * np.sqrt(1 / r), s * np.sqrt(r)
            axes[0].add_patch(plt.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, ec=cols[s], lw=1.5))
    axes[0].plot(cx, cy, "o", color="white", ms=6)
axes[0].set_title(f"9 anchors (3 scales x 3 ratios) at two of the {(messi.shape[0] // stride) * (messi.shape[1] // stride)} positions of a stride-{stride} grid")
axes[0].axis("off")
pipeline_diagram(["image", "backbone +\nFPN", "RPN: objectness +\nanchor offsets", "top-N proposals\n(NMS)", "RoIAlign\n7x7 per RoI",
                  "box head: class\n+ refined box", "per-class NMS"],
                 colors=["#f1f3f4", "#e8f0fe", "#fef7e0", "#fef7e0", "#e6f4ea", "#fce8e6", "#f1f3f4"],
                 notes=["", "shared features", "stage 1", "~1000 boxes", "", "stage 2", ""], ax=axes[1], fontsize=9)
axes[1].set_title("Faster R-CNN with FPN")
plt.suptitle("Figure 7.1 - Anchors and the two stages of Faster R-CNN", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# RoIAlign in one picture: bilinear samples of a fixed 7 x 7 grid inside an arbitrary (non-integer) box
feat = torch.tensor(cv2.cvtColor(messi, cv2.COLOR_BGR2GRAY), dtype=torch.float32)[None, None]
roi = torch.tensor([[0, 333.3, 283.7, 395.9, 339.2]])           # (batch index, x0, y0, x1, y1) around the ball
pooled = torchvision.ops.roi_align(feat, roi, output_size=(7, 7), spatial_scale=1.0, sampling_ratio=2, aligned=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(messi[..., ::-1]); x0, y0, x1, y1 = roi[0, 1:].tolist()
axes[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="#d93025", lw=2))
gx, gy = np.meshgrid(np.linspace(x0, x1, 15)[1::2], np.linspace(y0, y1, 15)[1::2])
axes[0].plot(gx, gy, ".", color="#fbbc05", ms=4); axes[0].set_xlim(290, 440); axes[0].set_ylim(342, 240); axes[0].set_title("RoI (red) and its 7 x 7 bin centres")
axes[1].imshow(pooled[0, 0].numpy(), cmap="gray"); axes[1].set_title("RoIAlign output: a fixed 7 x 7 grid, whatever the box size")
plt.suptitle("Figure 7.2 - RoIAlign turns any box into a fixed-size feature (here on the image itself)", fontweight="bold")
plt.tight_layout(); plt.show()

### Faster R-CNN inside out
We load torchvision's **Faster R-CNN with a MobileNetV3 + FPN backbone** (BSD licence, ~19 M parameters, fits a 4 GB GPU), trained on **COCO**
(80 classes). Instead of calling `model(images)` we run the stages ourselves to see the **proposals** of stage 1 and the **detections** of stage 2.

In [ ]:
from torchvision.models.detection import FasterRCNN_MobileNet_V3_Large_FPN_Weights, fasterrcnn_mobilenet_v3_large_fpn

frcnn, pretrained_ok = load_pretrained(fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT)
frcnn = frcnn.eval().to(DEVICE)
COCO = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT.meta["categories"]


@torch.no_grad()
def two_stage(img_bgr):
    x = [torch.tensor(img_bgr[..., ::-1].copy()).permute(2, 0, 1).float().div(255).to(DEVICE)]
    images, _ = frcnn.transform(x)                                   # resize + normalise
    feats = frcnn.backbone(images.tensors)                           # FPN feature maps
    proposals, _ = frcnn.rpn(images, feats)                          # stage 1
    dets, _ = frcnn.roi_heads(feats, proposals, images.image_sizes)  # stage 2
    dets = frcnn.transform.postprocess(dets, images.image_sizes, [img_bgr.shape[:2]])[0]
    sy, sx = img_bgr.shape[0] / images.image_sizes[0][0], img_bgr.shape[1] / images.image_sizes[0][1]
    props = proposals[0].cpu().numpy() * [sx, sy, sx, sy]
    return props, {k: v.cpu().numpy() for k, v in dets.items()}


t0 = time.perf_counter()
props, dets = two_stage(street)
dt = time.perf_counter() - t0
print(f"{'pretrained COCO weights' if pretrained_ok else 'RANDOM weights (no internet) - results are meaningless'} | "
      f"{len(props)} proposals, {len(dets['boxes'])} detections, {dt * 1000:.0f} ms on {DEVICE}")
fig, axes = plt.subplots(1, 2, figsize=(22, 7))
vis = street.copy()
for p in props[:150]:
    cv2.rectangle(vis, tuple(p[:2].astype(int)), tuple(p[2:].astype(int)), (255, 200, 0), 1)
axes[0].imshow(vis[..., ::-1]); axes[0].set_title("stage 1: the top 150 RPN proposals (class-agnostic)")
k = dets["scores"] > 0.5
lab = dets["labels"][k]
axes[1].imshow(partsim.draw_boxes(street, dets["boxes"][k], np.arange(k.sum()), dets["scores"][k],
                                  class_names=[COCO[i] for i in lab])[..., ::-1])
axes[1].set_title(f"stage 2: {k.sum()} detections with score > 0.5")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 7.3 - Faster R-CNN on vtest.avi: class-agnostic proposals, then classified and refined detections", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: score threshold on real images, and the closed-set problem
A COCO detector only knows its 80 classes. Try `baboon.jpg` and `squirrel_cls.jpg`: what does it say about animals that are not in COCO?

In [ ]:
extra = {"street (vtest.avi)": street, "messi5.jpg": messi, "baboon.jpg": load_sample("baboon.jpg"),
         "squirrel_cls.jpg": load_sample("squirrel_cls.jpg"), "conveyor scene (partsim)": scene}
cache = {k: two_stage(v)[1] for k, v in extra.items()}


def frcnn_demo(image="messi5.jpg", threshold=0.5):
    d = cache[image]; k = d["scores"] > threshold
    show(partsim.draw_boxes(extra[image], d["boxes"][k], np.arange(k.sum()), d["scores"][k], class_names=[COCO[i] for i in d["labels"][k]]),
         titles=[f"{image}: {k.sum()} detections above {threshold:.2f}"], figsize=(10, 7))


interact(frcnn_demo, image=list(extra), threshold=widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05))

## 8. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. COCO to YOLO
Convert COCO boxes `[N, 4]` = (x_min, y_min, w, h) in pixels to YOLO boxes (cx, cy, w, h) **normalised** by the image width and height.

In [ ]:
def coco_to_yolo(boxes, img_w, img_h):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    out = np.asarray(coco_to_yolo(np.array([[10, 20, 30, 40], [0, 0, 640, 480]], float), 640, 480), float)
    assert out.shape == (2, 4), f"expected shape (2, 4), got {out.shape}"
    assert np.allclose(out[0], [25 / 640, 40 / 480, 30 / 640, 40 / 480]), f"first box wrong: {out[0]}"
    assert np.allclose(out[1], [0.5, 0.5, 1, 1]), f"full-image box should be [0.5, 0.5, 1, 1], got {out[1]}"


check("E1 coco_to_yolo", _test_e1)

### E2. Non-maximum suppression
Return the **indices** of the boxes kept by greedy NMS, in decreasing score order. Use `box_iou` from Section 3 but not `torchvision.ops.nms`.

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    boxes = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [0, 0, 10, 10.5], [21, 19, 31, 30]], float)
    scores = np.array([0.9, 0.8, 0.7, 0.95, 0.3])
    keep = list(np.asarray(nms(boxes, scores, 0.5)).ravel())
    assert keep == [3, 2], f"expected [3, 2], got {keep}"
    r = np.random.default_rng(0)
    xy = r.uniform(0, 200, (60, 2)); wh = r.uniform(10, 50, (60, 2))
    bx = np.c_[xy, xy + wh]; sc = r.uniform(0, 1, 60)
    ref = torchvision.ops.nms(torch.tensor(bx), torch.tensor(sc), 0.4).numpy().tolist()
    got = list(np.asarray(nms(bx, sc, 0.4)).ravel())
    assert got == ref, "differs from torchvision.ops.nms on random boxes"


check("E2 nms", _test_e2)

### E3. Average precision
Given the list of detection `scores`, a boolean array `is_tp` and the number of ground-truth objects `n_gt`, return the **all-point interpolated AP**:
sort by score, compute cumulative precision and recall, take the precision envelope, and sum precision × recall steps. Do not call `ap_from_pr`.

In [ ]:
def average_precision(scores, is_tp, n_gt):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    assert abs(average_precision(np.array([0.9, 0.8, 0.7]), np.array([True, True, True]), 3) - 1.0) < 1e-9, "3 TPs for 3 objects -> AP 1"
    ap = average_precision(np.array([0.9, 0.8, 0.7, 0.6]), np.array([True, False, True, False]), 4)
    assert abs(ap - (0.25 * 1 + 0.25 * 2 / 3)) < 1e-9, f"expected 0.4167, got {ap:.4f}"
    ap = average_precision(np.array([0.6, 0.9, 0.8, 0.7]), np.array([False, True, False, True]), 4)
    assert abs(ap - (0.25 * 1 + 0.25 * 2 / 3)) < 1e-9, "you must sort by score first"
    assert average_precision(np.array([0.5]), np.array([False]), 2) == 0.0, "no TP -> AP 0"


check("E3 average_precision", _test_e3)

### E4. Another class (no self-check)
Train the HOG detector for `washer` (a ring, rotation invariant) and for `bolt` (elongated, any orientation). Which one reaches the higher AP, and
why is a single square window a bad model for bolts? How do modern detectors avoid this problem?

In [ ]:
# E4: your code here

## 9. Challenge: hard-negative mining and scale
1. Implement Dalal & Triggs' **hard-negative mining**: run the detector on new scenes, add every false positive with a score > −0.5 to the negatives,
   retrain, and compare AP. Does it help here? Explain why or why not by looking at *which* errors remain (localisation vs background).
2. Generate scenes with objects at 0.7–1.4× scale (resize the scene) and add an **image pyramid** to the detector. Plot AP vs number of pyramid levels
   and run time.

In [ ]:
# Your challenge code here

## 10. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": "A YOLO label line reads '2 0.5 0.5 0.25 0.1' for a 640 x 480 image. What is the box width in pixels?",
        "options": [
            '0.25',
            '120',
            '160',
            '64',
        ],
        "answer": '1065b097d0',
        "explain": 'w = 0.25 * 640 = 160 px.',
    },
    {
        "q": 'Two 10 x 10 boxes overlap in a 5 x 10 region. What is their IoU?',
        "options": [
            '0.25',
            '0.33',
            '0.5',
            '0.67',
        ],
        "answer": 'c4f829bc5c',
        "explain": '50 / (100 + 100 - 50) = 1/3.',
    },
    {
        "q": 'What does non-maximum suppression remove?',
        "options": [
            'Low-contrast pixels',
            'Duplicate detections of the same object',
            'Small objects',
            'Background classes',
        ],
        "answer": '60e940e235',
        "explain": 'It keeps the best box and removes others that overlap it strongly.',
    },
    {
        "q": 'A detector finds 8 of 10 objects and makes 2 false detections. Precision and recall?',
        "options": [
            'P = 0.8, R = 0.8',
            'P = 0.8, R = 1.0',
            'P = 1.0, R = 0.8',
            'P = 0.6, R = 0.8',
        ],
        "answer": 'c3db064a13',
        "explain": 'P = 8/10, R = 8/10.',
    },
    {
        "q": 'Why does COCO mAP average over IoU thresholds 0.50-0.95?',
        "options": [
            'To reward precise localisation',
            'To speed up evaluation',
            'To count small objects twice',
            'Because NMS needs it',
        ],
        "answer": '8a1814b0ae',
        "explain": 'A box that barely reaches IoU 0.5 scores poorly at higher thresholds.',
    },
    {
        "q": 'What does the Region Proposal Network predict at each anchor?',
        "options": [
            'The class of the object',
            'An objectness score and box offsets',
            'A segmentation mask',
            'The image label',
        ],
        "answer": '147b6ea99e',
        "explain": 'Classification into the C classes happens in stage 2.',
    },
    {
        "q": 'Why did RoIAlign replace RoI pooling?',
        "options": [
            'It is faster',
            'It avoids rounding the box to the feature grid, improving localisation',
            'It removes the need for anchors',
            'It adds colour',
        ],
        "answer": '9bd61c0581',
        "explain": 'Bilinear sampling keeps sub-pixel accuracy.',
    },
    {
        "q": "A COCO detector labels a squirrel as a 'bird'. This illustrates...",
        "options": [
            'overfitting',
            'the closed-set assumption',
            'NMS failure',
            'low IoU',
        ],
        "answer": '2f3f61d541',
        "explain": 'It must choose among its 80 training classes; open-vocabulary detectors (Session B) relax this.',
    },
], title='Week 08A quiz')

## 11. Exit ticket (reflection)
1. Your part detector has AP50 = 0.95 but AP75 = 0.40. What does this tell you, and does it matter for a pick-and-place robot?
2. Why does a sliding-window detector become slow when you add scales and aspect ratios, and how does an RPN avoid that?
3. For a production line, would you choose the score threshold to maximise precision or recall? Justify with the cost of each error.
4. Our HOG detector fires on the round holes of washers and flanges (Figure 4.2). Which training step of Dalal & Triggs targets exactly this error?

## 12. What's new (2025–26)

> - Two-stage detectors are no longer the accuracy leaders for general objects — transformer detectors (**DETR** family, **RT-DETR**, **D-FINE**,
>   **RF-DETR**) and one-stage **YOLO** models dominate the COCO leaderboard (Session B) — but Faster/Mask R-CNN remain strong, well-understood
>   baselines for small datasets and high-resolution inspection, and are the basis of many industrial tools.
> - **Evaluation beyond mAP**: *LRP* and *TIDE* break the error into classification, localisation, duplicate, background and missed errors,
>   which is far more useful for debugging than one number.
> - **Open-vocabulary** detectors (Grounding DINO, OWLv2, YOLO-World, YOLOE) remove the closed-set limitation shown in Section 7 (Session B).

## 13. References and further exploration

**Seminal papers**
- Viola, P. & Jones, M. (2001). *Rapid object detection using a boosted cascade of simple features.* CVPR. [doi:10.1109/CVPR.2001.990517](https://doi.org/10.1109/CVPR.2001.990517)
- Dalal, N. & Triggs, B. (2005). *Histograms of oriented gradients for human detection.* CVPR. [doi:10.1109/CVPR.2005.177](https://doi.org/10.1109/CVPR.2005.177)
- Felzenszwalb, P. et al. (2010). *Object detection with discriminatively trained part-based models.* IEEE TPAMI 32(9). [doi:10.1109/TPAMI.2009.167](https://doi.org/10.1109/TPAMI.2009.167)
- Girshick, R. et al. (2014). *Rich feature hierarchies for accurate object detection* (R-CNN). CVPR. [arXiv:1311.2524](https://arxiv.org/abs/1311.2524)
- Girshick, R. (2015). *Fast R-CNN.* ICCV. [arXiv:1504.08083](https://arxiv.org/abs/1504.08083)
- Ren, S. et al. (2015). *Faster R-CNN: Towards real-time object detection with region proposal networks.* NeurIPS. [arXiv:1506.01497](https://arxiv.org/abs/1506.01497)
- Lin, T.-Y. et al. (2017). *Feature pyramid networks for object detection.* CVPR. [arXiv:1612.03144](https://arxiv.org/abs/1612.03144)
- He, K. et al. (2017). *Mask R-CNN* (RoIAlign). ICCV. [arXiv:1703.06870](https://arxiv.org/abs/1703.06870)
- Bodla, N. et al. (2017). *Soft-NMS — improving object detection with one line of code.* ICCV. [arXiv:1704.04503](https://arxiv.org/abs/1704.04503)
- Everingham, M. et al. (2010). *The PASCAL Visual Object Classes (VOC) challenge.* IJCV 88. [doi:10.1007/s11263-009-0275-4](https://doi.org/10.1007/s11263-009-0275-4)
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common objects in context.* ECCV. [arXiv:1405.0312](https://arxiv.org/abs/1405.0312)
- Bolya, D. et al. (2020). *TIDE: A general toolbox for identifying object detection errors.* ECCV. [arXiv:2008.08115](https://arxiv.org/abs/2008.08115)

**Textbooks, courses and tools**
- Szeliski (2022), §6.3 (object detection). [szeliski.org/Book](https://szeliski.org/Book/) · Prince (2023), *Understanding Deep Learning*, Ch. 10. [udlbook.github.io](https://udlbook.github.io/udlbook/)
- torchvision detection models and the *TorchVision Object Detection Finetuning Tutorial*. [pytorch.org/vision](https://pytorch.org/vision/stable/models.html#object-detection-instance-segmentation-and-person-keypoint-detection)
- COCO detection evaluation (metric definitions). [cocodataset.org/#detection-eval](https://cocodataset.org/#detection-eval)

**Images:** `vtest.avi`, `messi5.jpg`, `baboon.jpg`, `squirrel_cls.jpg` from the OpenCV sample data; conveyor scenes generated by `partsim.py`.

## 14. Key takeaways
- Detection outputs a **variable-length list** of (box, class, score); know your **box format** (xyxy, COCO xywh, YOLO normalised cxcywh).
- **IoU** measures overlap; small objects are very sensitive to localisation error.
- Classical detectors = **classifier × every window** (HOG + SVM); **NMS** removes duplicates.
- Evaluate with **precision–recall** and **AP**; COCO **mAP50-95** also rewards accurate boxes.
- **Faster R-CNN**: shared CNN features, an **RPN** with anchors (stage 1), **RoIAlign** + a box head (stage 2); **FPN** handles scale.

**Next session (08B):** one-stage detectors (**YOLO**), transformer detectors (**DETR**, **RT-DETR**) and **open-vocabulary** detection.